# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlahimani/Flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:

from google.colab import userdata
import duckdb

# connect to database
con = duckdb.connect()

con.sql("INSTALL httpfs; LOAD httpfs;")

# Use token saved in Colab
my_token = userdata.get("token_dataaccess")

con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{my_token}'
);
""")

print("Connected successfully.")

Connected successfully.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

raw_data = con.sql("""
    SELECT content_hash_id, client_hash_id, report_date,
           gsc_clicks, gsc_impressions, gsc_avg_position,
           ga4_engaged_sessions, ga4_sessions, ga4_data_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()

raw_data = raw_data[raw_data["ga4_data_available"] == True]

raw_data = raw_data[raw_data["report_date"] < "2026-03-16"]

raw_data = raw_data.drop(columns=["report_date"])

feature_vector = raw_data.groupby(["content_hash_id", "client_hash_id"]).agg({
    "gsc_clicks": "sum",           # total clicks over the 15 days
    "gsc_impressions": "sum",      # total impressions over the 15 days
    "gsc_avg_position": "mean",    # AVERAGE position, not summed
    "ga4_engaged_sessions": "sum", # total engaged sessions
    "ga4_sessions": "sum"          # total sessions
})

feature_vector = feature_vector.reset_index()

feature_vector["ctr_1h"] = feature_vector["gsc_clicks"] / feature_vector["gsc_impressions"]

feature_vector["engagement_rate_1h"] = feature_vector["ga4_engaged_sessions"] / feature_vector["ga4_sessions"]

print("Number of rows:", len(feature_vector))
print("Number of columns:", len(feature_vector.columns))
feature_vector.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of rows: 50746
Number of columns: 9


,content_hash_id,client_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,ga4_sessions,ctr_1h,engagement_rate_1h
0,content_0000a348850eb1fc,client_3ffa76342f366962,0,0,NaN,0,1,NaN,0.0
1,content_00032be2df0005ca,client_fef1a8f436438636,2,140,6.771232,0,42,0.014286,0.0
2,content_00036470b65bb8a6,client_ba65e80a1116ae41,0,0,NaN,0,1,NaN,0.0
3,content_00039f4c7a954114,client_157ffe4d4a595515,0,4,4.000000,0,1,0.000000,0.0
4,content_0007b250f3b58e86,client_d211cb07b9059bab,0,0,NaN,0,1,NaN,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Missing values per column:")
print(feature_vector.isna().sum())

print("\nAny negative clicks?", (feature_vector["gsc_clicks"] < 0).any())
print("Any negative impressions?", (feature_vector["gsc_impressions"] < 0).any())


Missing values per column:
index                       0
content_hash_id             0
client_hash_id              0
gsc_clicks                  0
gsc_impressions             0
gsc_avg_position        14449
ga4_engaged_sessions        0
ga4_sessions                0
ctr_1h                  14449
engagement_rate_1h          0
dtype: int64

Any negative clicks? False
Any negative impressions? False


In [10]:
# Step 1: make a flag column that remembers which rows had zero impressions
feature_vector["had_impressions"] = feature_vector["gsc_impressions"] > 0

# Step 2: fill ctr_1h missing values with 0
# (0 impressions = 0 clicks possible = 0% CTR makes sense)
feature_vector["ctr_1h"] = feature_vector["ctr_1h"].fillna(0)

# Step 3: fill gsc_avg_position missing values with a big number
# (no impressions basically means "not ranking anywhere", so we use
# a large placeholder like 100 to represent "very bad / not ranking")
feature_vector["gsc_avg_position"] = feature_vector["gsc_avg_position"].fillna(100)

# Step 4: double check nothing is missing anymore
print("Missing values after fix:")
print(feature_vector.isna().sum())

Missing values after fix:
content_hash_id         0
client_hash_id          0
gsc_clicks              0
gsc_impressions         0
gsc_avg_position        0
ga4_engaged_sessions    0
ga4_sessions            0
ctr_1h                  0
engagement_rate_1h      0
had_impressions         0
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
label_data = con.sql("""
    SELECT content_hash_id, client_hash_id, gsc_clicks
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE report_date >= '2026-03-16'
""").df()

label_data = label_data.groupby(["content_hash_id", "client_hash_id"]).sum().reset_index()

label_data = label_data.rename(columns={"gsc_clicks": "clicks_2h"})

combined = feature_vector.merge(label_data, on=["content_hash_id", "client_hash_id"])

correlations = combined[["gsc_clicks", "gsc_impressions", "gsc_avg_position", "ctr_1h", "engagement_rate_1h"]].corrwith(combined["clicks_2h"])

print("Correlation between each feature and FUTURE clicks (clicks_2h):\n")
print(correlations)

Correlation between each feature and FUTURE clicks (clicks_2h):

gsc_clicks            0.886960
gsc_impressions       0.555020
gsc_avg_position     -0.050541
ctr_1h               -0.027184
engagement_rate_1h    0.021003
dtype: float64


## 4. What I excluded and why
*The list of fields you refused to use — with one line of why each.*

1.trend_direction, trend_pct — I didn't use these because they're basically a repackaged version of the thing I'm trying to predict. Using them would be like cheating on a test by looking at the answer key.

2.health_score, recommended_action, action_type (product flags) — I left these out because they're scores someone else already calculated using their own system. If I used them,my model would just be copying an existing decision, not actually learning anything new from the real data.

3.clicks_2h (the second-half clicks I calculated in Section 3) — I only used this to TEST for leakage, not as an actual feature. It's literally the future outcome, so including it as a real feature would make my results fake/too good to be true.

4.content_hash_id, client_hash_id — I only use these to match rows together (like a lookup ID), never as a real input. They're just random ID codes, they don't mean anything on their own, so a model can't actually learn from them.

5.Rows where ga4_data_available is FALSE — I removed these rows completely, because a "0" in that case doesn't mean "zero engagement," it just means "we don't have that data yet."Keeping them in would make my numbers look worse than they actually are.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [yes ] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes ] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.